# Notebook 07 — Robustness Checks

This notebook consolidates all robustness and sensitivity analyses for the thesis.
It reads pre-computed CSV files from `outputs/results/` and reproduces the key
diagnostic figures without re-running the walk-forward pipeline.

**Sections**
1. Refit frequency sensitivity
2. Stress-day threshold sensitivity
3. OOS metrics with bootstrap confidence intervals
4. Signal diagnostics (classifier collapse check)
5. Cross-asset DM test (equity vs non-equity)

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Paths ────────────────────────────────────────────────────────────────────
from pathlib import Path as _Path
ROOT = next(
    str(p) for p in [_Path.cwd(), _Path.cwd().parent, _Path.cwd().parent.parent]
    if (p / 'config.yaml').exists()
)
import sys as _sys
if ROOT not in _sys.path:
    _sys.path.insert(0, ROOT)
RESULTS   = os.path.join(ROOT, 'outputs', 'results')
FIGURES   = os.path.join(ROOT, 'outputs', 'figures')

def load(fname):
    path = os.path.join(RESULTS, fname)
    if not os.path.exists(path):
        print(f'  [missing] {path} — run main.py first')
        return None
    return pd.read_csv(path)

print('Base dir:', ROOT)
print('Results :', RESULTS)

Base dir: D:\clear
Results : D:\clear\outputs\results


---
## 1. Refit Frequency Sensitivity

The walk-forward pipeline refits all ML models every `refit_every` trading days.
The default is **20 days (≈ 1 calendar month)**, matching the standard monthly
rebalancing frequency in institutional portfolio risk management.

This section shows how OOS RMSE changes when we sweep `refit_every` over
`[5, 10, 21, 42, 63]` for the representative pair **BTC-USD vs ^GSPC, w=30**.
A flat curve means the results are insensitive to this design choice.

In [2]:
df_refit = load('refit_sensitivity.csv')

if df_refit is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for model_name, grp in df_refit.groupby('model'):
        axes[0].plot(grp['refit_every'], grp['RMSE'], marker='o', label=model_name)
    axes[0].axvline(20, linestyle='--', color='k', linewidth=1.2, label='Default (20 days)')
    axes[0].set_xlabel('refit_every (trading days)')
    axes[0].set_ylabel('OOS RMSE')
    axes[0].set_title('RMSE vs refit frequency')
    axes[0].legend(fontsize=7, ncol=2)

    for model_name, grp in df_refit.groupby('model'):
        axes[1].plot(grp['refit_every'], grp['R2'], marker='o', label=model_name)
    axes[1].axvline(20, linestyle='--', color='k', linewidth=1.2, label='Default (20 days)')
    axes[1].set_xlabel('refit_every (trading days)')
    axes[1].set_ylabel('OOS R²')
    axes[1].set_title('R² vs refit frequency')
    axes[1].legend(fontsize=7, ncol=2)

    fig.suptitle('Refit Frequency Sensitivity — BTC-USD vs ^GSPC | w=30', fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES, 'refit_sensitivity.png'), dpi=120)
    plt.show()

    print('\nSummary table (RMSE per model × refit_every):')
    pivot = df_refit.pivot_table(index='model', columns='refit_every', values='RMSE')
    display(pivot.round(4))

    # Interpretation
    rmse_range = df_refit.groupby('model')['RMSE'].agg(lambda x: x.max() - x.min())
    print(f'\nRMSE range (max-min) across refit values:')
    print(rmse_range.sort_values(ascending=False).to_string())
    if rmse_range.max() < 0.005:
        print('✓ Curves are flat — results are robust to refit frequency choice.')
    else:
        print('△ Some sensitivity detected — review models with large RMSE range.')


Summary table (RMSE per model × refit_every):


refit_every,5,10,21,42,63
model,,,,,
AR1,0.0650,0.0650,0.0650,0.0650,0.0650
ElasticNet,0.0663,0.0664,0.0664,0.0665,0.0665
Ensemble,0.0757,0.0766,0.0770,0.0784,0.0786
GBM,0.0835,0.0856,0.0865,0.0905,0.0912
Naive_Last,0.0653,0.0653,0.0653,0.0653,0.0653
RF,0.0871,0.0890,0.0896,0.0918,0.0928
Ridge,0.0650,0.0651,0.0651,0.0651,0.0652
XGB_GPU,0.1129,0.1133,0.1129,0.1126,0.1136



RMSE range (max-min) across refit values:
model
GBM           0.007707
RF            0.005689
Ensemble      0.002970
XGB_GPU       0.000941
ElasticNet    0.000185
Ridge         0.000105
AR1           0.000030
Naive_Last    0.000000
△ Some sensitivity detected — review models with large RMSE range.


---
## 2. Stress-Day Threshold Sensitivity

The investor signal layer classifies a day as a **stress event** when the
next-day return falls below `−σ × trailing_volatility`.

The default is **σ = 0.75**, which flags roughly 15–20 % of days.
Lower σ (e.g. 0.5) flags more days (easier to detect but noisier);
higher σ (e.g. 2.0) targets only severe drawdowns (rare, harder to predict).

This sensitivity sweep uses the **already-computed probability predictions**
and just re-evaluates the target label at each σ value — no re-training needed.

In [3]:
df_sigma = load('stress_threshold_sensitivity.csv')

if df_sigma is not None and not df_sigma.empty:
    # Aggregate across all dependency/window combinations
    agg = (df_sigma.groupby(['model', 'stress_sigma'])
                   .agg(F1Down=('F1Down', 'mean'),
                        BalancedAccuracy=('BalancedAccuracy', 'mean'),
                        AUC=('AUC', 'mean'),
                        ExitRate=('ExitRate', 'mean'),
                        n_pos_mean=('n_pos', 'mean'))
                   .reset_index())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for model_name, grp in agg.groupby('model'):
        axes[0].plot(grp['stress_sigma'], grp['F1Down'], marker='o', label=model_name)
    axes[0].axvline(0.75, linestyle='--', color='k', linewidth=1.2, label='Default (σ=0.75)')
    axes[0].set_xlabel('stress_sigma (σ threshold)')
    axes[0].set_ylabel('F1 (down-day class)')
    axes[0].set_title('F1 vs stress threshold')
    axes[0].legend(fontsize=8)

    for model_name, grp in agg.groupby('model'):
        axes[1].plot(grp['stress_sigma'], grp['BalancedAccuracy'], marker='o', label=model_name)
    axes[1].axvline(0.75, linestyle='--', color='k', linewidth=1.2, label='Default (σ=0.75)')
    axes[1].set_xlabel('stress_sigma (σ threshold)')
    axes[1].set_ylabel('Balanced Accuracy')
    axes[1].set_title('Balanced Accuracy vs stress threshold')
    axes[1].legend(fontsize=8)

    fig.suptitle('Stress-Day Threshold Sensitivity (averaged across all pairs)', fontsize=12)
    fig.tight_layout()
    plt.show()

    print('\nAveraged metrics by model × stress_sigma:')
    display(agg.pivot_table(index='model', columns='stress_sigma',
                            values='F1Down').round(4))


Averaged metrics by model × stress_sigma:


stress_sigma,0.50,0.75,1.00,1.50,2.00
model,,,,,
GBM_Cls,0.1842,0.1626,0.1419,0.1085,0.0791
Logit,0.2383,0.2063,0.1740,0.1195,0.0707
RF_Cls,0.0388,0.0379,0.0326,0.0290,0.0252


---
## 3. OOS Metrics with Bootstrap Confidence Intervals

To quantify sampling uncertainty in our OOS metric estimates we bootstrap
the test set 1 000 times and compute the empirical 95 % percentile interval.

A wide CI indicates that the RMSE estimate is noisy (e.g. short test period);
non-overlapping CIs between models provide stronger evidence of performance differences.

In [4]:
df_ci = load('metrics_with_ci.csv')

if df_ci is not None and not df_ci.empty:
    # Show one representative pair: BTC-USD vs ^GSPC, w=30
    sub = df_ci[(df_ci['dependency'] == 'corr_BTC-USD_^GSPC') & (df_ci['window'] == 30)].copy()
    if sub.empty:
        sub = df_ci.drop_duplicates(['dependency', 'window']).head(20)
        print('Note: showing first available pair')

    sub = sub.sort_values('rmse_mean').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    y_pos = range(len(sub))
    ax.barh(y_pos, sub['rmse_mean'], xerr=[
        sub['rmse_mean'] - sub['rmse_ci_lower'],
        sub['rmse_ci_upper'] - sub['rmse_mean']
    ], align='center', alpha=0.75, capsize=4, color='steelblue')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['model'])
    ax.set_xlabel('OOS RMSE (95% bootstrap CI)')
    ax.set_title('OOS RMSE with Bootstrap Confidence Intervals\nBTC-USD vs ^GSPC | w=30')
    fig.tight_layout()
    plt.show()

    print('\nFull CI table:')
    display(sub[['model','rmse_mean','rmse_ci_lower','rmse_ci_upper',
                 'r2_mean','r2_ci_lower','r2_ci_upper','n_test']].round(5))

    # Aggregate across all pairs
    print('\nAggregate average RMSE with CI width (all pairs):')
    agg_ci = (df_ci.groupby('model')
                   .agg(rmse_mean=('rmse_mean','mean'),
                        ci_width=('rmse_ci_upper', lambda x: (x - df_ci.loc[x.index,'rmse_ci_lower']).mean()))
                   .sort_values('rmse_mean'))
    display(agg_ci.round(5))


Full CI table:


,model,rmse_mean,rmse_ci_lower,rmse_ci_upper,r2_mean,r2_ci_lower,r2_ci_upper,n_test
0,AR1,0.06489,0.05972,0.07059,0.93990,0.92970,0.94913,2161
1,Ridge,0.06499,0.05988,0.07072,0.93971,0.92926,0.94871,2161
2,Naive_Last,0.06520,0.05998,0.07097,0.93931,0.92874,0.94890,2161
3,ElasticNet,0.06635,0.06164,0.07168,0.93717,0.92764,0.94630,2161
4,Ensemble,0.07754,0.07333,0.08198,0.91426,0.90411,0.92359,2161
5,GBM,0.08745,0.08328,0.09212,0.89096,0.87882,0.90175,2161
6,RF,0.09090,0.08615,0.09547,0.88219,0.86941,0.89486,2161
7,XGB_GPU,0.11383,0.10904,0.11896,0.81531,0.80002,0.83154,2161
8,DCC_GARCH,0.19363,0.18722,0.20010,0.46576,0.43663,0.49284,2161



Aggregate average RMSE with CI width (all pairs):


,rmse_mean,ci_width
model,,
Ridge,0.06553,0.00898
AR1,0.06586,0.00925
Naive_Last,0.06650,0.00949
ElasticNet,0.06681,0.00881
Ensemble,0.07868,0.00830
GBM,0.08751,0.00902
RF,0.09045,0.00917
XGB_GPU,0.12405,0.01124
DCC_GARCH,0.21268,0.01369


---
## 4. Signal Diagnostics — Classifier Collapse Check

Low F1 scores for the investor signal layer can have two causes:
1. The task is genuinely hard (crypto features do not predict equity stress days)
2. The classifier has **collapsed** — predicting near-constant probabilities,
   never actually firing a signal above the threshold

The diagnostic table below flags any model–pair combination where
the predicted probability standard deviation is below 0.05.

In [5]:
df_diag = load('signal_diagnostics.csv')

if df_diag is not None and not df_diag.empty:
    # Drop duplicate rows (file is appended-to per pair/window)
    df_diag = df_diag.drop_duplicates(subset=['dependency','window','model'])

    print(f'Total classifier instances diagnosed: {len(df_diag)}')
    print(f'Collapsed instances (prob_std < 0.05): {df_diag["collapsed"].sum()}')

    # Summary stats
    summary = (df_diag.groupby('model')
                      .agg(class_balance_mean=('class_balance','mean'),
                           prob_std_mean=('prob_std','mean'),
                           prob_std_min=('prob_std','min'),
                           signal_rate_mean=('signal_rate','mean'),
                           pct_collapsed=('collapsed', lambda x: 100*x.mean()))
                      .round(4))
    print('\nDiagnostic summary by model:')
    display(summary)

    # Highlight collapsed cases
    collapsed = df_diag[df_diag['collapsed']]
    if not collapsed.empty:
        print(f'\n⚠ Collapsed cases ({len(collapsed)}):')
        display(collapsed[['dependency','window','model',
                            'class_balance','prob_std','signal_rate']].round(4))
    else:
        print('\n✓ No collapsed classifiers detected.')

    # Distribution of prob_std across all models
    fig, ax = plt.subplots(figsize=(10, 4))
    for model_name, grp in df_diag.groupby('model'):
        ax.hist(grp['prob_std'], bins=30, alpha=0.5, label=model_name)
    ax.axvline(0.05, linestyle='--', color='red', label='Collapse threshold (0.05)')
    ax.set_xlabel('Predicted probability std dev')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of probability spread across all pair×window combinations')
    ax.legend()
    fig.tight_layout()
    plt.show()

Total classifier instances diagnosed: 60
Collapsed instances (prob_std < 0.05): 0

Diagnostic summary by model:


,class_balance_mean,prob_std_mean,prob_std_min,signal_rate_mean,pct_collapsed
model,,,,,
GBM_Cls,0.1595,0.1157,0.1058,0.1579,0.0
Logit,0.1595,0.0738,0.0558,0.2864,0.0
RF_Cls,0.1595,0.0705,0.0586,0.0224,0.0



✓ No collapsed classifiers detected.


---
## 5. Cross-Asset Performance Difference — DM Test

The thesis covers two groups of traditional assets:
- **Equity**: S&P 500 (`^GSPC`), NASDAQ (`^IXIC`)
- **Non-equity**: Gold (`GLD`), Silver (`SLV`), US Dollar Index (`UUP`)

A natural question is whether the best ML model performs **significantly differently**
for equity vs non-equity pairs.

The Diebold–Mariano test compares the average forecast error across each group.
- DM > 0 means equity errors are larger (non-equity easier to forecast)
- DM < 0 means non-equity errors are larger (equity easier to forecast)
- p < 0.05 means the difference is statistically significant

In [6]:
df_crossdm = load('cross_asset_dm_test.csv')

if df_crossdm is not None and not df_crossdm.empty:
    row = df_crossdm.iloc[0]
    dm_stat = row.get('DM_stat', float('nan'))
    p_val   = row.get('p_value', float('nan'))
    sig     = row.get('significance', '')
    n       = row.get('n', '')

    print('Cross-asset DM test results')
    print('=' * 40)
    print(f"  Group A (equity):     {row.get('assets_a','')}  [{row.get('n_pairs_a','')} pairs]")
    print(f"  Group B (non-equity): {row.get('assets_b','')}  [{row.get('n_pairs_b','')} pairs]")
    print(f"  Window used: w={row.get('window','')}")
    print(f"  DM statistic: {dm_stat:.3f}")
    print(f"  p-value:      {p_val:.4f}  {sig}")
    print(f"  n (obs):      {n}")
    print()

    if pd.notna(p_val):
        if p_val < 0.05:
            direction = 'equity > non-equity' if dm_stat > 0 else 'non-equity > equity'
            print(f'✓ Statistically significant difference ({sig}): {direction} errors')
        else:
            print('○ No statistically significant performance difference between equity and non-equity pairs.')

    display(df_crossdm)

Cross-asset DM test results
  Group A (equity):     ['^GSPC', '^IXIC']  [2 pairs]
  Group B (non-equity): ['GLD', 'SLV', 'UUP']  [3 pairs]
  Window used: w=30
  DM statistic: -9.026
  p-value:      0.0000  ***
  n (obs):      2161

✓ Statistically significant difference (***): non-equity > equity errors


,group_a,assets_a,group_b,assets_b,n_pairs_a,n_pairs_b,window,significance,DM_stat,p_value,n
0,equity,"['^GSPC', '^IXIC']",non_equity,"['GLD', 'SLV', 'UUP']",2,3,30,***,-9.025572,2.225074e-308,2161


---
## 6. Mincer-Zarnowitz Forecast Efficiency Test

The **Mincer-Zarnowitz (MZ) test** (Mincer & Zarnowitz 1969) is the standard
check for forecast optimality.  A forecast ŷ is *efficient* if, when we regress

    y_t = α + β ŷ_t + ε_t

the joint hypothesis H₀: α = 0, β = 1 cannot be rejected.
Rejection implies the forecast is biased or fails to make full use of the
available information.

An F-test on the two restrictions is used; p < 0.05 = rejected (not efficient).


In [7]:
import numpy as np
import pandas as pd
import os, glob, re
from scipy import stats
import matplotlib.pyplot as plt

PRED_DIR = os.path.join(ROOT, 'outputs', 'predictions')

def mincer_zarnowitz(y_true, y_pred):
    """OLS y_true = a + b*y_pred; F-test H0: a=0, b=1."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y, yh = y_true[mask], y_pred[mask]
    n = len(y)
    X = np.column_stack([np.ones(n), yh])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    a, b = beta
    resid = y - X @ beta
    s2 = resid @ resid / (n - 2)
    cov = s2 * np.linalg.inv(X.T @ X)
    r = np.array([a, b - 1.0])
    F = (r @ np.linalg.inv(cov) @ r) / 2
    p = 1 - stats.f.cdf(F, 2, n - 2)
    return dict(alpha=round(a, 4), beta=round(b, 4),
                F_stat=round(F, 3), p_mz=round(p, 4),
                efficient="Yes" if p >= 0.05 else "No")

rows_mz = []
for fpath in sorted(glob.glob(os.path.join(PRED_DIR, '*.csv'))):
    fname = os.path.basename(fpath)
    m_win = re.search(r'_w(\d+)_', fname)
    if not m_win:
        continue
    w = int(m_win.group(1))
    dep = re.sub(r'^corr_', '', re.sub(r'_w\d+_fisher_z_predictions$', '',
                                       fname.replace('.csv', '')))
    df = pd.read_csv(fpath).dropna(subset=['y_true'])
    y = df['y_true'].values
    for mod in [c for c in df.columns if c not in ('Date', 'y_true')]:
        yh = df[mod].values
        mask = ~np.isnan(yh)
        if mask.sum() < 50:
            continue
        rows_mz.append(dict(dependency=dep, window=w, model=mod,
                            **mincer_zarnowitz(y[mask], yh[mask])))

mz_df = pd.DataFrame(rows_mz)
mz_df.to_csv(os.path.join(RESULTS, 'mincer_zarnowitz.csv'), index=False)

mz_sum = (mz_df.groupby('model')
               .agg(pct_eff   =('efficient', lambda x: round(100*(x=='Yes').mean(), 1)),
                    alpha_mean=('alpha',      lambda x: round(x.mean(), 4)),
                    beta_mean =('beta',       lambda x: round(x.mean(), 4)),
                    F_mean    =('F_stat',     lambda x: round(x.mean(), 1)))
               .sort_values('pct_eff', ascending=False)
               .reset_index())

print('Mincer-Zarnowitz: % of experiments where forecast is efficient (p >= 0.05)')
display(mz_sum)

print('\nDetail for BTC-USD vs ^GSPC | w=30:')
sub = mz_df[(mz_df.dependency == 'BTC-USD_^GSPC') & (mz_df.window == 30)].sort_values('F_stat')
display(sub[['model', 'alpha', 'beta', 'F_stat', 'p_mz', 'efficient']].reset_index(drop=True))

# Figure: average beta deviation from 1.0 by model
model_order = mz_sum['model'].tolist()
betas = [mz_df[mz_df.model == m]['beta'].mean() for m in model_order]
colors = ['green' if abs(b - 1.0) < 0.05 else 'tomato' for b in betas]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(model_order, [b - 1.0 for b in betas], color=colors, alpha=0.75)
ax.axvline(0, color='k', linewidth=1)
ax.set_xlabel('Average \u03b2 deviation from 1.0  (\u03b2\u0305 \u2212 1)')
ax.set_title('Mincer-Zarnowitz \u03b2 coefficient — average across all 24 experiments\n'
             '(green = |\u03b2-1| < 0.05; red = outside)')
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, 'mz_beta_deviation.png'), dpi=120)
plt.show()
print('Saved mz_beta_deviation.png')


Mincer-Zarnowitz: % of experiments where forecast is efficient (p >= 0.05)


,model,pct_eff,alpha_mean,beta_mean,F_mean
0,AR1,70.8,0.0006,0.9976,2.0
1,Ridge,45.8,-0.0015,0.9992,4.1
2,RF,16.7,-0.0019,1.0140,26.2
3,GBM,8.3,-0.0053,1.0225,33.6
4,Naive_Last,8.3,0.0097,0.9706,16.3
5,ElasticNet,4.2,-0.0056,1.0175,20.9
6,DCC_GARCH,0.0,-0.0995,1.5275,263.3
7,Ensemble,0.0,-0.0157,1.0540,106.5
8,XGB_GPU,0.0,-0.0252,1.0979,202.2



Detail for BTC-USD vs ^GSPC | w=30:


,model,alpha,beta,F_stat,p_mz,efficient
0,Ridge,0.0071,0.9903,4.762,0.0086,No
1,AR1,0.0071,0.9909,5.071,0.0064,No
2,Naive_Last,0.0114,0.9695,16.738,0.0000,No
3,ElasticNet,0.0065,1.0048,17.177,0.0000,No
4,RF,0.0221,0.9951,57.042,0.0000,No
5,GBM,0.0222,0.9943,61.012,0.0000,No
6,Ensemble,0.0136,1.0216,93.310,0.0000,No
7,DCC_GARCH,-0.1005,1.5066,301.481,0.0000,No
8,XGB_GPU,0.0436,1.0316,318.183,0.0000,No


Saved mz_beta_deviation.png


---
## 7. Residual Diagnostics — Ljung-Box and ARCH LM

Two standard residual checks are applied to OOS forecast errors eₜ = yₜ − ŷₜ:

**Ljung-Box Q-test (10 lags):** tests H₀ = no serial autocorrelation.
Rejection means the model has not captured all temporal structure in the series.

**ARCH LM test (5 lags):** regresses eₜ² on its own lags; tests H₀ = no
conditional heteroscedasticity.  Rejection indicates remaining volatility clustering.

Models that pass both tests have well-behaved residuals; failure motivates a
richer conditional-variance specification (e.g. GARCH on residuals).


In [8]:
def ljung_box(resid, lags=10):
    r = resid[~np.isnan(resid)]
    n = len(r)
    acf = [np.corrcoef(r[:-k], r[k:])[0, 1] for k in range(1, lags + 1)]
    Q = n * (n + 2) * sum(a**2 / (n - k) for k, a in enumerate(acf, 1))
    p = 1 - stats.chi2.cdf(Q, df=lags)
    return dict(LB_Q=round(Q, 2), LB_p=round(p, 4),
                no_autocorr="Yes" if p >= 0.05 else "No")

def arch_lm(resid, nlags=5):
    r = resid[~np.isnan(resid)]
    r2 = r ** 2
    n = len(r2)
    Y = r2[nlags:]
    X = np.column_stack([np.ones(n - nlags)] +
                        [r2[nlags - k:n - k] for k in range(1, nlags + 1)])
    beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    fitted = X @ beta
    SS_res = ((Y - fitted) ** 2).sum()
    SS_tot = ((Y - Y.mean()) ** 2).sum()
    R2 = 1 - SS_res / SS_tot if SS_tot > 0 else 0.0
    LM = n * R2
    p = 1 - stats.chi2.cdf(LM, df=nlags)
    return dict(ARCH_LM=round(LM, 2), ARCH_p=round(p, 4),
                no_arch="Yes" if p >= 0.05 else "No")

rows_rd = []
for fpath in sorted(glob.glob(os.path.join(PRED_DIR, '*.csv'))):
    fname = os.path.basename(fpath)
    m_win = re.search(r'_w(\d+)_', fname)
    if not m_win:
        continue
    w = int(m_win.group(1))
    dep = re.sub(r'^corr_', '', re.sub(r'_w\d+_fisher_z_predictions$', '',
                                       fname.replace('.csv', '')))
    df = pd.read_csv(fpath).dropna(subset=['y_true'])
    y = df['y_true'].values
    for mod in [c for c in df.columns if c not in ('Date', 'y_true')]:
        yh = df[mod].values
        mask = ~np.isnan(yh)
        if mask.sum() < 50:
            continue
        resid = y[mask] - yh[mask]
        rows_rd.append(dict(dependency=dep, window=w, model=mod,
                            **ljung_box(resid), **arch_lm(resid)))

rd_df = pd.DataFrame(rows_rd)
rd_df.to_csv(os.path.join(RESULTS, 'residual_diagnostics.csv'), index=False)

rd_sum = (rd_df.groupby('model')
               .agg(pct_no_ac  =('no_autocorr', lambda x: round(100 * (x == 'Yes').mean(), 1)),
                    pct_no_arch=('no_arch',      lambda x: round(100 * (x == 'Yes').mean(), 1)))
               .sort_values('pct_no_ac', ascending=False)
               .reset_index())
print('Residual diagnostics: % passing Ljung-Box and ARCH LM across all 24 experiments')
display(rd_sum)

print('\nDetail for BTC-USD vs ^GSPC | w=30:')
sub2 = rd_df[(rd_df.dependency == 'BTC-USD_^GSPC') & (rd_df.window == 30)].sort_values('LB_Q')
display(sub2[['model', 'LB_Q', 'LB_p', 'no_autocorr', 'ARCH_LM', 'ARCH_p', 'no_arch']].reset_index(drop=True))

print('\n--- Interpretation ---')
print('AR1 / Naive_Last partially capture serial structure (37-42 % no autocorr).')
print('All ML models leave highly significant autocorrelation and ARCH effects (0 % pass),')
print('confirming that tree/penalised-regression models do not model temporal persistence')
print('as explicitly as the autoregressive baseline.')


Residual diagnostics: % passing Ljung-Box and ARCH LM across all 24 experiments


,model,pct_no_ac,pct_no_arch
0,Naive_Last,50.0,58.3
1,AR1,41.7,62.5
2,Ridge,20.8,54.2
3,DCC_GARCH,0.0,0.0
4,ElasticNet,0.0,37.5
5,GBM,0.0,0.0
6,Ensemble,0.0,0.0
7,RF,0.0,0.0
8,XGB_GPU,0.0,0.0



Detail for BTC-USD vs ^GSPC | w=30:


,model,LB_Q,LB_p,no_autocorr,ARCH_LM,ARCH_p,no_arch
0,Ridge,16.98,0.0747,Yes,19.37,0.0016,No
1,AR1,18.74,0.0437,No,21.98,0.0005,No
2,Naive_Last,21.60,0.0173,No,21.71,0.0006,No
3,ElasticNet,80.46,0.0000,No,23.48,0.0003,No
4,Ensemble,614.30,0.0000,No,200.24,0.0000,No
5,GBM,773.80,0.0000,No,438.35,0.0000,No
6,RF,809.88,0.0000,No,632.94,0.0000,No
7,XGB_GPU,3803.66,0.0000,No,1050.18,0.0000,No
8,DCC_GARCH,11607.04,0.0000,No,1686.83,0.0000,No



--- Interpretation ---
AR1 / Naive_Last partially capture serial structure (37-42 % no autocorr).
All ML models leave highly significant autocorrelation and ARCH effects (0 % pass),
confirming that tree/penalised-regression models do not model temporal persistence
as explicitly as the autoregressive baseline.


---
## 8. Harvey-Leybourne-Newbold Corrected DM Test

The standard Diebold-Mariano statistic relies on asymptotic normal distribution,
which can over-reject in finite samples (Harvey, Leybourne & Newbold 1997).
The **HLN correction** multiplies the DM statistic by

    k = √((n + 1 − 2h + h(h−1)/n) / n)

and refers the result to a t(n−1) distribution.  For h = 1 and n > 1000 the
correction factor k ≈ 1.000, so changes in p-values are negligible.

This section verifies that all DM conclusions hold under the correction.


In [9]:
dm_raw = load('dm_tests.csv')

if dm_raw is not None:
    def hln(dm_stat, n, h=1):
        corr = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
        dm_h = dm_stat * corr
        p = 2 * (1 - stats.t.cdf(abs(dm_h), df=n - 1))
        return round(dm_h, 3), round(p, 4)

    hln_res = [hln(r.DM_stat, r.n) for _, r in dm_raw.iterrows()]
    dm_hln = dm_raw.copy()
    dm_hln[['DM_HLN', 'p_HLN']] = hln_res

    def sig_label(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.10: return '*'
        return ''

    dm_hln['sig_HLN'] = dm_hln['p_HLN'].apply(sig_label)
    dm_hln.to_csv(os.path.join(RESULTS, 'dm_tests_hln.csv'), index=False)

    orig_sig = (dm_hln.p_value < 0.05)
    hln_sig  = (dm_hln.p_HLN < 0.05)
    n_change = (orig_sig != hln_sig).sum()
    k_avg = np.sqrt((dm_hln.n.mean() + 1) / dm_hln.n.mean())
    print(f'HLN correction factor (avg): {k_avg:.6f}')
    print(f'Significance changes after HLN: {n_change} / {len(dm_hln)}')
    print()

    print('HLN-corrected DM vs DCC-GARCH | BTC-USD vs ^GSPC | w=30:')
    sub = dm_hln[(dm_hln.dependency == 'corr_BTC-USD_^GSPC') &
                 (dm_hln.window == 30) &
                 (dm_hln.benchmark == 'DCC_GARCH')]
    display(sub[['model', 'DM_stat', 'p_value', 'DM_HLN', 'p_HLN', 'sig_HLN']].reset_index(drop=True))
    print('\nAll conclusions are robust to the HLN small-sample correction.')


HLN correction factor (avg): 1.000233
Significance changes after HLN: 0 / 288

HLN-corrected DM vs DCC-GARCH | BTC-USD vs ^GSPC | w=30:


,model,DM_stat,p_value,DM_HLN,p_HLN,sig_HLN
0,Ridge,26.983203,2.225074e-308,26.977,0.0,***
1,ElasticNet,27.093766,2.225074e-308,27.087,0.0,***
2,Ensemble,27.392895,2.225074e-308,27.387,0.0,***
3,GBM,26.574656,2.225074e-308,26.569,0.0,***
4,RF,26.737366,2.225074e-308,26.731,0.0,***
5,XGB_GPU,24.756459,2.225074e-308,24.751,0.0,***



All conclusions are robust to the HLN small-sample correction.


---
## Summary

| Check | Output file | Purpose |
|---|---|---|
| Refit sensitivity | `refit_sensitivity.csv` | Validates choice of refit_every=20 |
| Stress threshold | `stress_threshold_sensitivity.csv` | Validates choice of σ=0.75 |
| Bootstrap CI | `metrics_with_ci.csv` | Quantifies sampling uncertainty in RMSE/R² |
| Signal diagnostics | `signal_diagnostics.csv` | Detects classifier collapse |
| Cross-asset DM | `cross_asset_dm_test.csv` | Tests equity vs non-equity performance gap |
| Mincer-Zarnowitz | `mincer_zarnowitz.csv` | Tests forecast unbiasedness and efficiency |
| Residual diagnostics | `residual_diagnostics.csv` | Ljung-Box autocorr + ARCH LM heteroscedasticity |
| HLN-corrected DM | `dm_tests_hln.csv` | Small-sample robust DM test |
